# New Dataset Preparation - Expanded

### Packages

In [1]:
# Basic Package Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# sklearn
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import RandomForestClassifier
from sklearn.decomposition import PCA
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics import f1_score
from sklearn.metrics import mean_squared_error, r2_score

# imblearn
from imblearn.over_sampling import RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler

# Non-basic package imports
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import requests

# Packages I don't understand
from fcd_torch import FCD
import rdkit
from collections import Counter
import gc
import pickle

# Add the Python_files directory to the Python path
import sys
import os
sys.path.append(os.path.join(os.path.dirname(os.getcwd()), 'Python_files'))

# Now you can import your modules
# import functions_enc as f
import function_depot as fd

# Data Processing

### DF5 Upload

GROUP STATISTICS 
Group                     Total Spectra   Unique SMILES  
LTQ-Orbitrap-negative     121             23             
LTQ-Orbitrap-positive     615             70             
LTQ-negative              13              10             
LTQ-positive              12              2              
Other-negative            12              8              
Other-positive            79              25             
Q-Orbitrap-negative       1400            252            
Q-Orbitrap-positive       2065            378            
Q-TOF-negative            235             56             
Q-TOF-positive            1098            221            
QQQ-negative              162             34             
QQQ-positive              286             85             

Total across all groups: 6098            678      

The code that produced this

Print group statistics for unique SMILES and spectra counts
print("=== GROUP STATISTICS ===")
print(f"{'Group':<25} {'Total Spectra':<15} {'Unique SMILES':<15}")
print("-" * 55)

for group in sorted(df5['Group'].unique()):
    group_data = df5[df5['Group'] == group]
    total_spectra = len(group_data)
    unique_smiles = group_data['SMILES_spectra'].nunique()
    
    print(f"{group:<25} {total_spectra:<15} {unique_smiles:<15}")

print(f"\nTotal across all groups: {len(df5):<15} {df5['SMILES_spectra'].nunique():<15}")

Groups included: ['Q-Orbitrap-positive' 'Q-TOF-positive' 'LTQ-Orbitrap-positive']
Unique SMILES: 485

In [2]:
# We are working with the June 25 dataset, with the Morgan Fingerprints and cannonical SMILES included
df5 = pd.read_csv("/home/dlipsey/MITLincolnLabs/MIT_LL_data/MIT_LL_data5.csv")
# print(df5.shape)
# df5.head()

# First order of business is to standardize our SMILES column. We want to use canonical smiles rather than SMILES_spectra but 
# we will keep the column name SMILES_spectra for consistency with previous code
df5 = df5.drop('SMILES_spectra', axis=1) # Drop
df5 = df5.rename(columns={'canonical_smiles': 'SMILES_spectra'}) # Rename
cols = df5.columns.tolist()
cols.remove('SMILES_spectra') 
df5 = df5[['SMILES_spectra'] + cols] # Move to front

# Next we want to standardize the Ionization column
# print(df5["Ionization_Mode"].unique()) # Check unique values
df5["Ionization_Mode"] = df5["Ionization_Mode"].replace("'Positive'", "'positive'") # Fix capitaliztion
df5 = df5[df5["Ionization_Mode"] != "'N/A'"] # Remove N/A 
# print(df5["Ionization_Mode"].unique()) # Check unique values

# Remove single quotes from all columns
df5 = df5.applymap(lambda x: x.replace("'", "") if isinstance(x, str) else x)

# Select specific groups for subset
selected_groups = ['Q-Orbitrap-positive', 'Q-TOF-positive', 'LTQ-Orbitrap-positive']

# Create subset with only selected groups
df5_subset = df5[df5['Group'].isin(selected_groups)]

print(df5_subset.shape)
df5_subset.head()

(3778, 18)


,SMILES_spectra,CAS,Molecular_Formula,Total_Exact_Mass,Precursor_m/z,Precursor_Type,Spectrum,Ionization_Mode,Instrument_Type,Instrument_Name,Collision_Energy,SMILES_tox_vals,Response_Modifier,Response,Response_Unit,Group,fp,filtered_fp
0,C#CCN(C)Cc1ccccc1,555-57-7,C11H13N,159.104799416,160.1121,[M+H]+,63.0228:0.177223 65.0386:5.629055 68.0495:0.45...,positive,LC-ESI-QFT,Q Exactive Orbitrap Thermo Scientific,90 (nominal),C#CCN(C)Cc1ccccc1,,273.642508,mg/kg,Q-Orbitrap-positive,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
1,C#CCN(C)Cc1ccccc1,555-57-7,C11H13N,159.104799416,160.1121,[M+H]+,63.0228:0.125979 65.0386:2.113734 68.0495:0.68...,positive,LC-ESI-QFT,Q Exactive Orbitrap Thermo Scientific,75 (nominal),C#CCN(C)Cc1ccccc1,,273.642508,mg/kg,Q-Orbitrap-positive,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
2,C#CCN(C)Cc1ccccc1,555-57-7,C11H13N,159.104799416,160.1121,[M+H]+,56.0496:0.115017 65.0386:0.970445 68.0495:1.03...,positive,LC-ESI-QFT,Q Exactive Orbitrap Thermo Scientific,60 (nominal),C#CCN(C)Cc1ccccc1,,273.642508,mg/kg,Q-Orbitrap-positive,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
3,C#CCN(C)Cc1ccccc1,555-57-7,C11H13N,159.104799416,160.1121,[M+H]+,51.0229:0.102992 56.0495:0.143820 65.0386:0.67...,positive,LC-ESI-QFT,Q Exactive Orbitrap Thermo Scientific,45 (nominal),C#CCN(C)Cc1ccccc1,,273.642508,mg/kg,Q-Orbitrap-positive,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
4,C#CCN(C)Cc1ccccc1,555-57-7,C11H13N,159.104799416,160.1121,[M+H]+,56.0496:0.482623 65.0385:0.377829 68.0495:2.59...,positive,LC-ESI-QFT,Q Exactive Orbitrap Thermo Scientific,30 (nominal),C#CCN(C)Cc1ccccc1,,273.642508,mg/kg,Q-Orbitrap-positive,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."


### Dataframe Creation - Added to MIT_LL_data directory

In [3]:
# DF5 SUBSET DATAFRAME
# df5_subset.to_csv('/home/dlipsey/MITLincolnLabs/MIT_LL_data/df5_subset.csv', index=False)


In [4]:
# SPECTRA DATAFRAME
# Create dataframe with spectra using spectrum_string_to_dataframe
df5_spectra = fd.spectrum_string_to_dataframe(df5_subset, spectrum_col='Spectrum', smiles_col='SMILES_spectra')

# Add Group and Response columns by mapping directly from df5_subset
# Create dictionaries for faster lookup
smiles_to_group = df5_subset.set_index('SMILES_spectra')['Group'].to_dict()
smiles_to_response = df5_subset.set_index('SMILES_spectra')['Response'].to_dict()

# Map the values directly
df5_spectra['Group'] = df5_spectra['SMILES_spectra'].map(smiles_to_group)
df5_spectra['Response'] = df5_spectra['SMILES_spectra'].map(smiles_to_response)

print("=== SPECTRA DATAFRAME ===")
print(f"Shape: {df5_spectra.shape}")
print(f"Unique SMILES: {df5_spectra['SMILES_spectra'].nunique()}")
print(f"Columns: {list(df5_spectra.columns[:3])} ... {list(df5_spectra.columns[-3:])}")  # Show first and last few columns


# # CHEMNET EMBEDDINGS
# # Create ChemNet embeddings dataframe using get_chemnet_emb_from_smiles
# unique_smiles = df5_subset['SMILES_spectra'].unique().tolist()
# print(f"Getting ChemNet embeddings for {len(unique_smiles)} unique SMILES...")

# # Get embeddings dictionary
# embeddings_dict = fd.get_chemnet_emb_from_smiles(unique_smiles)

# # Convert to dataframe format
# embeddings_data = []
# for smiles, embedding in embeddings_dict.items():
#     if embedding != 'unknown':  # Skip unknown embeddings
#         row = {'SMILES_spectra': smiles}
#         # Add each embedding dimension as a separate column
#         for i, emb_val in enumerate(embedding):
#             row[f'Embedding Float {i}'] = emb_val
#         embeddings_data.append(row)

# df5_chemnet = pd.DataFrame(embeddings_data)

# print("\n=== EMBEDDINGS DATAFRAME ===")
# print(f"Shape: {df5_chemnet.shape}")
# print(f"Unique SMILES: {df5_chemnet['SMILES_spectra'].nunique()}")
# print(f"Embedding dimensions: {df5_chemnet.shape[1] - 1}")  # -1 for SMILES column
# df5_chemnet.head()


# # MORGAN FINGERPRINTS
# # Create dataframe with just SMILES and Morgan fingerprints using expand_fingerprints_to_matrix
# df5_morganfp= fd.expand_fingerprints_to_matrix(df5_subset, smiles_col='SMILES_spectra', fp_col='fp')

# print("\n=== FINGERPRINTS DATAFRAME ===")
# print(f"Shape: {df5_morganfp.shape}")
# print(f"Unique SMILES: {df5_morganfp['SMILES_spectra'].nunique()}")
# df5_morganfp.head()

=== SPECTRA DATAFRAME ===
Shape: (3778, 64443)
Unique SMILES: 485
Columns: ['SMILES_spectra', 29.0112, 30.032] ... ['index_id', 'Group', 'Response']


In [ ]:
# # Save the datafames to data folder
# df5_spectra = df5_spectra.to_csv('/home/dlipsey/MITLincolnLabs/MIT_LL_data/df5_spectra.csv', index=False)
# df5_chemnet = df5_chemnet.to_csv('/home/dlipsey/MITLincolnLabs/MIT_LL_data/df5_chemnet.csv', index=False)
# df5_morganfp = df5_morganfp.to_csv('/home/dlipsey/MITLincolnLabs/MIT_LL_data/df5_morganfp.csv', index=False)

# # Load dataframes into notebook - Beware, these can take a while to load
# df5_spectra = pd.read_csv('/home/dlipsey/MITLincolnLabs/MIT_LL_data/df5_spectra.csv')
# df5_chemnet = pd.read_csv('/home/dlipsey/MITLincolnLabs/MIT_LL_data/df5_chemnet.csv')
# df5_morganfp = pd.read_csv('/home/dlipsey/MITLincolnLabs/MIT_LL_data/df5_morganfp.csv')


KeyboardInterrupt: 

In [ ]:
import os
# Verify that df5_subset was created
file_path = '/home/dlipsey/MITLincolnLabs/MIT_LL_data/df5_subset.csv'
if os.path.exists(file_path):
    print(f"File exists at: {file_path}")
    print(f"File size: {os.path.getsize(file_path)} bytes")
else:
    print(f"File does NOT exist at: {file_path}")
# Verify that df5_spectra was created
import os
file_path = '/home/dlipsey/MITLincolnLabs/MIT_LL_data/df5_spectra.csv'
if os.path.exists(file_path):
    print(f"File exists at: {file_path}")
    print(f"File size: {os.path.getsize(file_path)} bytes")
else:
    print(f"File does NOT exist at: {file_path}")

# Verify that df5_morganfp was created
import os
file_path = '/home/dlipsey/MITLincolnLabs/MIT_LL_data/df5_morganfp.csv'
if os.path.exists(file_path):
    print(f"File exists at: {file_path}")
    print(f"File size: {os.path.getsize(file_path)} bytes")
else:
    print(f"File does NOT exist at: {file_path}")

# Verify that df5_chement was created
import os
file_path = '/home/dlipsey/MITLincolnLabs/MIT_LL_data/df5_chemnet.csv'
if os.path.exists(file_path):
    print(f"File exists at: {file_path}")
    print(f"File size: {os.path.getsize(file_path)} bytes")
else:
    print(f"File does NOT exist at: {file_path}")

# Binning

In [ ]:
df5_spectra.head()


,SMILES_spectra,29.0112,30.032,30.0323,31.01686,31.54035,38.5076,39.0214,39.0215,39.02194,...,1939.240845,1965.805054,1966.380615,1982.848389,2000.461914,2000.942627,index_id,Group,Response,log_response
0,C#CCN(C)Cc1ccccc1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0,Q-Orbitrap-positive,273.642508,5.611823
1,C#CCN(C)Cc1ccccc1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1,Q-Orbitrap-positive,273.642508,5.611823
2,C#CCN(C)Cc1ccccc1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,2,Q-Orbitrap-positive,273.642508,5.611823
3,C#CCN(C)Cc1ccccc1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,3,Q-Orbitrap-positive,273.642508,5.611823
4,C#CCN(C)Cc1ccccc1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,4,Q-Orbitrap-positive,273.642508,5.611823


In [ ]:
# Define your parameters
bin_sizes = [0.05, 0.1, 0.5, 1, 2, 5, 10, 25, 50, 100, 200, 500, 1000]
thresholds = [0.001, 0.005, 0.01, 0.05, 0.1, 0.5, 1, 2, 5, 10, 50, 100]
save_directory = "/home/dlipsey/MITLincolnLabs/MIT_LL_data/grid_search_dataframes"

# Create all datasets
# all_datasets = fd.binning_loop(df5_spectra, df5_subset, bin_sizes, thresholds, save_directory)

Creating all binned and thresholded datasets...


TypeError: '>' not supported between instances of 'str' and 'float'